In [1]:
import faiss
print(f"FAISS version: {faiss.__version__}")

FAISS version: 1.13.2


In [2]:
import faiss
import numpy as np

d = 384  # dimension des embeddings
index = faiss.IndexFlatIP(d)  # ou IndexFlatIP(d)
print(f"Index trained: {index.is_trained}")  # True (pas d'entraînement requis)

Index trained: True


In [3]:
import faiss
import numpy as np

# Paramètres
d = 384  # dimension des vecteurs
nb = 1000  # nombre de vecteurs dans la base
nq = 5  # nombre de requêtes

# Générer des données aléatoires
np.random.seed(42)
xb = np.random.random((nb, d)).astype('float32')  # base
xq = np.random.random((nq, d)).astype('float32')  # requêtes

# Créer et remplir l'index
index = faiss.IndexFlatIP(d)
index.add(xb)
print(f"Indexed vectors: {index.ntotal}")
# Vecteurs indexés: 1000

# Recherche des 4 plus proches voisins
k = 4
D, I = index.search(xq, k)

# D = distances, I = indices
print(f"Indices top-{k}: {I[0]}")
print(f"Distances: {D[0]}")

Indexed vectors: 1000
Indices top-4: [658 566 644 927]
Distances: [103.85547  103.5397   103.389404 103.08659 ]


In [4]:
print(xq[0])

[0.19547069 0.44427058 0.34957284 0.28842476 0.6941563  0.97408503
 0.8191521  0.95503205 0.20729186 0.66037184 0.40130648 0.30080166
 0.02190265 0.65735453 0.9126402  0.5318811  0.62670237 0.4148118
 0.3821981  0.6256478  0.12632935 0.06190041 0.46079126 0.27828175
 0.5025176  0.2099911  0.43433574 0.41992024 0.7593989  0.28598356
 0.26131532 0.38978428 0.10834735 0.4408531  0.4511251  0.3343987
 0.96519667 0.59312034 0.41686475 0.7656429  0.3881665  0.01985529
 0.7330141  0.10365756 0.82178646 0.6109769  0.48496392 0.63137645
 0.09282248 0.49328595 0.22620662 0.91759974 0.0977911  0.94873816
 0.4325803  0.8095923  0.97823507 0.29185292 0.7698389  0.918608
 0.54016274 0.83668107 0.06459563 0.03098822 0.33935896 0.01091252
 0.01118551 0.62183887 0.83397686 0.27445897 0.12787592 0.10102398
 0.9271005  0.10127661 0.3917331  0.2318559  0.15558212 0.57254946
 0.94535017 0.52659243 0.48204145 0.6106985  0.20641509 0.01562798
 0.5837766  0.78531754 0.06398486 0.25737807 0.0853917  0.05460775

In [18]:
from BM25_and_embeddings import chunks_from_markdown_files
from sentence_transformers import SentenceTransformer

chunks = chunks_from_markdown_files()
# for chunk in chunks[:10]:
#   print(chunk)

print("Loading model...")
model = SentenceTransformer(model_name_or_path = "sentence-transformers/all-MiniLM-L6-v2")
print("Generating embeddings...")
sentences = [chunk["chunk"] for chunk in chunks]
embeddings = model.encode(inputs = sentences, show_progress_bar = False, convert_to_numpy = True)

c:\Users\yjaquier\.conda\envs\ollama\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1881.60it/s]


Generating embeddings...


In [32]:
from constants import CACHE_DIR

print(f"Embeddings length: {len(embeddings)}")
# Paramètres
d = 384  # dimension des vecteurs
# Créer et remplir l'index
index = faiss.IndexFlatIP(d)
index.add(embeddings)
print(f"Indexed vectors: {index.ntotal}")
# faiss.write_index(index, str(CACHE_DIR / "faiss_index.bin"))
# index = faiss.read_index(str(CACHE_DIR / "faiss_index.bin"))
# print(f"Index rechargé: {index.ntotal} vecteurs")

for item in zip(chunks[:5], embeddings[:5]):
    print("-" * 80)
    print(f"File: {item[0]['file']}, Chunk: {item[0]['chunk']}...")
    print("Embedding sample:", item[1][:10], "\n")

Embeddings length: 5220
Indexed vectors: 5220
--------------------------------------------------------------------------------
File: skills-main\db\admin\backup-recovery.md, Chunk: # oracle rman backup and recovery...
Embedding sample: [-0.12931053  0.02086154 -0.04167901 -0.06984172  0.0931142  -0.03627417
  0.05703097  0.02853256  0.00105044 -0.03780094] 

--------------------------------------------------------------------------------
File: skills-main\db\admin\backup-recovery.md, Chunk: ## overview

recovery manager (rman) is oracle's primary tool for database backup, restore, and recovery operations. it provides block-level backup integrity checking, compression, encryption, incremental backups, and tight integration with the oracle database engine. rman eliminates the need for manual backup scripting and ensures consistent backups even for an open database.

understanding backup and recovery is the single most critical skill for any oracle dba. a database that cannot be recovered

In [38]:
super_total = [ {"file": chunk["file"], "chunk": chunk["chunk"], "embedding": embedding} for chunk, embedding in zip(chunks, embeddings)]
print(super_total[0])
print(type(super_total[0]["embedding"]))

{'file': 'skills-main\\db\\admin\\backup-recovery.md', 'chunk': '# oracle rman backup and recovery', 'embedding': array([-1.29310533e-01,  2.08615381e-02, -4.16790135e-02, -6.98417202e-02,
        9.31142047e-02, -3.62741724e-02,  5.70309721e-02,  2.85325572e-02,
        1.05043827e-03, -3.78009379e-02,  8.23156983e-02,  3.04468721e-02,
        3.28327380e-02, -1.22781461e-02,  5.23426430e-03, -1.36360321e-02,
       -1.25220358e-01,  2.25244928e-02,  2.01306697e-02,  6.18309788e-02,
       -5.21989800e-02,  6.12897724e-02,  1.59300752e-02, -4.22693044e-03,
        3.18871439e-02,  5.33025712e-02,  5.01150638e-02,  4.21513058e-02,
       -9.30759311e-03, -3.76319736e-02,  6.08872715e-03,  6.69535249e-02,
        2.69755181e-02, -6.87036142e-02,  5.61429933e-02,  1.11674398e-01,
       -2.30130032e-02,  2.79075336e-02, -3.63355130e-02, -4.40656394e-02,
       -9.28473398e-02, -3.72648649e-02, -1.24054132e-02, -3.53455842e-02,
       -2.45437846e-02, -4.84320521e-02, -2.30274834e-02,  2.

In [ ]:
import importlib
import BM25_and_embeddings

importlib.reload(BM25_and_embeddings)
from BM25_and_embeddings import bm25_score

query = "How to generate an AWR report in HTML format ?"
results, scores = bm25_score(query, chunks, top_k=10)
print(results)
print(scores)

d:\Yannick\Development\python\ollama\BM25_and_embeddings.py:18: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  """


[[{'file': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'chunk': '### diagnose a recent performance problem\n\n```sql\n-- list snapshots to find the relevant time window\nawr list snapshots\n\n-- generate an html report for that window (e.g., snapshots 142 to 145)\nawr create html 142 145\n```\n\nopen the resulting `.html` file in a browser for the full formatted report including top sql, wait events, load profile, and instance efficiency metrics.'}
  {'file': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'chunk': '## best practices\n\n- run `awr list snapshots` before generating a report to confirm the snapshot ids exist and span the time window you want.\n- use html format for human review; use text format when the output needs to be parsed programmatically or sent in plain-text email.\n- keep snapshot intervals short (15–30 minutes) during active problem investigation so you can isolate the exact period when performance degraded.\n- the default awr snapshot interval is 60 minutes. for fine-gra

In [84]:
print(results[0, 1])

for i in range(results.shape[1]):
  result, score = results[0, i], scores[0, i]
  print(f"Result {i+1}: {result}, Score: {score}")

{'file': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'chunk': '## best practices\n\n- run `awr list snapshots` before generating a report to confirm the snapshot ids exist and span the time window you want.\n- use html format for human review; use text format when the output needs to be parsed programmatically or sent in plain-text email.\n- keep snapshot intervals short (15–30 minutes) during active problem investigation so you can isolate the exact period when performance degraded.\n- the default awr snapshot interval is 60 minutes. for fine-grained baselining during a test, take manual snapshots at shorter intervals with `awr create snapshot`.\n- run awr reports on a non-production replica when possible — generating large awr reports against a heavily loaded production database consumes i/o and memory.\n\n---'}
Result 1: {'file': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'chunk': '### diagnose a recent performance problem\n\n```sql\n-- list snapshots to find the relevant time window\nawr 

In [86]:
print(chunks[0])

{'file': 'skills-main\\db\\admin\\backup-recovery.md', 'chunk': '# oracle rman backup and recovery'}
